In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


In [2]:
df = pd.read_csv('../After_EDA_data/Light_text.csv')

In [3]:
df

,uid,profile,anime_uid,score,scores,text
0,255938,DesolatePsyche,34096,8,"{'Overall': '8', 'Story': '8', 'Animation': '8...",first things first my reviews system is explai...
1,259117,baekbeans,34599,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",let me start off by saying that made in abyss ...
2,253664,skrn,28891,7,"{'Overall': '7', 'Story': '7', 'Animation': '9...",10 it is great especially the actions during t...
3,8254,edgewalker00,2904,9,"{'Overall': '9', 'Story': '9', 'Animation': '9...",story taking place 1 yr from where season 1 tr...
4,291149,aManOfCulture99,4181,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",kyoto animations greatest strength is being ab...
...,...,...,...,...,...,...
192107,240067,Unicorn819,1281,9,"{'Overall': '9', 'Story': '5', 'Animation': '1...",ok this anime is pretty old but here s the bac...
192108,285777,ShizzoSVH,1281,9,"{'Overall': '9', 'Story': '7', 'Animation': '9...",the dub for this anime is made this anime a fu...
192109,286904,AlluMan96,1281,3,"{'Overall': '3', 'Story': '3', 'Animation': '1...",some might argue that doing a review of a show...
192110,287903,AgentK300,1281,10,"{'Overall': '10', 'Story': '3', 'Animation': '...",absolutely hilarious i accidentally came acros...


In [4]:
threshold = 6
df['target'] = (df['score'] >= threshold).astype(int)

In [5]:
print("Class distribution before any sampling:")
print(df['target'].value_counts())

Class distribution before any sampling:
target
1    158840
0     33272
Name: count, dtype: int64


In [6]:
df_majority = df[df['target']==1]
df_minority = df[df['target']==0]

In [9]:
majority_downsampled = df_majority.sample(n=len(df_minority), random_state=42)

df_balanced = pd.concat([majority_downsampled, df_minority]).sample(frac=1, random_state=42)

print("\nClass distribution after moderate undersampling:")
print(df_balanced['target'].value_counts())


Class distribution after moderate undersampling:
target
0    33272
1    33272
Name: count, dtype: int64


In [10]:
X = df_balanced['text']
y = df_balanced['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [11]:
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1,2),
    lowercase=True,
    stop_words=None
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [12]:
clf = LogisticRegression(
    solver='saga',
    max_iter=1000,
    class_weight='balanced',
    C=1.0
)

clf.fit(X_train_tfidf, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [13]:
y_pred = clf.predict(X_test_tfidf)
y_prob = clf.predict_proba(X_test_tfidf)[:,1]

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))



Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      6655
           1       0.89      0.88      0.88      6654

    accuracy                           0.88     13309
   macro avg       0.88      0.88      0.88     13309
weighted avg       0.88      0.88      0.88     13309



In [14]:
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:

[[5939  716]
 [ 818 5836]]


In [15]:
roc_auc = roc_auc_score(y_test, y_prob)
print(f"\nROC-AUC Score: {roc_auc:.4f}")


ROC-AUC Score: 0.9501


In [16]:
feature_names = tfidf.get_feature_names_out()
coefs = clf.coef_[0]

top_pos_idx = coefs.argsort()[-20:][::-1]
top_neg_idx = coefs.argsort()[:20]

In [17]:
print("\nTop Positive Words:")
for i in top_pos_idx:
    print(f"{feature_names[i]} : {coefs[i]:.4f}")


Top Positive Words:
great : 7.0156
amazing : 5.7002
perfect : 5.4421
definitely : 4.7427
10 10 : 4.7093
enjoyed : 4.3323
awesome : 3.9718
the best : 3.9240
masterpiece : 3.9235
beautiful : 3.8677
well : 3.8147
unique : 3.7003
loved : 3.6834
excellent : 3.6119
perfectly : 3.5615
outstanding : 3.5453
enjoyable : 3.5376
love : 3.5192
bit : 3.3993
fantastic : 3.3428


In [18]:
print("\nTop Negative Words:")
for i in top_neg_idx:
    print(f"{feature_names[i]} : {coefs[i]:.4f}")


Top Negative Words:
boring : -7.0726
bad : -7.0445
worst : -6.9625
poor : -6.5855
nothing : -6.3739
horrible : -6.1374
mediocre : -6.1085
the worst : -5.8572
poorly : -5.5032
terrible : -5.4960
worse : -4.9328
bland : -4.8076
no : -4.6237
waste : -4.5244
only : -4.3138
pathetic : -4.2494
awful : -4.1180
fails : -4.0094
any : -3.9214
decent : -3.8288
